In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!rm -rf /content/dataset
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/dataset.zip -d /content/dataset

In [ ]:
import csv
import os
import time
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from tqdm.auto import tqdm

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
dataset_root = r"./dataset"
save_path = "/content/drive/MyDrive/Train_ResNet"
BEST_MODEL_PATH = os.path.join(save_path, "best.pt")
CHECKPOINT_PATH = os.path.join(save_path, "checkpoint_last.pt")
LOG_PATH = os.path.join(save_path, "training_log.csv")

# ── Chọn kiến trúc ResNet ─────────────────────────────────────────────────────
# Có thể chọn resnet18, resnet34, resnet50, resnet101, resnet152
RESNET_VARIANT = "resnet50"
RESNET_DROPOUT_CONFIG = {
    "resnet18": 0.25,
    "resnet34": 0.30,
    "resnet50": 0.35,
    "resnet101": 0.45,
    "resnet152": 0.50,
}
RESNET_WD_CONFIG = {
    "resnet18": 1e-2,
    "resnet34": 1.5e-2,
    "resnet50": 2e-2,
    "resnet101": 3e-2,
    "resnet152": 3e-2,
}
RESNET_LR0_CONFIG = {
    "resnet18": 8e-4,
    "resnet34": 6e-4,
    "resnet50": 5e-4,
    "resnet101": 3e-4,
    "resnet152": 2e-4,
}
RESNET_LR_CONFIG = {
    "resnet101": {
        "stem": 0.05,
        "layer1": 0.05,
        "layer2": 0.10,
        "layer3": 0.20,
        "layer4": 0.60,
        "fc": 1.00,
    },
    "resnet152": {
        "stem": 0.05,
        "layer1": 0.05,
        "layer2": 0.10,
        "layer3": 0.20,
        "layer4": 0.60,
        "fc": 1.00,
    },
    "resnet50": {
        "stem": 0.05,
        "layer1": 0.08,
        "layer2": 0.15,
        "layer3": 0.30,
        "layer4": 0.80,
        "fc": 1.00,
    },
    "resnet34": {
        "stem": 0.05,
        "layer1": 0.10,
        "layer2": 0.20,
        "layer3": 0.40,
        "layer4": 1.00,
        "fc": 1.00,
    },
    "resnet18": {
        "stem": 0.05,
        "layer1": 0.10,
        "layer2": 0.20,
        "layer3": 0.40,
        "layer4": 1.00,
        "fc": 1.00,
    },
}

# ── Hyperparameters ───────────────────────────────────────────────────────────
EPOCHS = 80
IMGSZ = 384
BATCH = 32
ACCUMULATION_STEPS = 2
LR0 = RESNET_LR0_CONFIG.get(RESNET_VARIANT, 3e-4)
WEIGHT_DECAY = RESNET_WD_CONFIG.get(RESNET_VARIANT, 3e-2)

PATIENCE = 12
NUM_WORKERS = 2
WARMUP_EPOCHS = 3
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

DROPOUT = RESNET_DROPOUT_CONFIG.get(RESNET_VARIANT, 0.45)

# ── Class counts (train set, alphabet order) ──────────────────────────────────
# battery, biological, cardboard, clothes, e-waste, glass, metal, paper, plastic, shoes
CLASS_COUNTS = [974, 1501, 1978, 1718, 1776, 2364, 3144, 2248, 2220, 1198]

# ── Mixup / CutMix ────────────────────────────────────────────────────────────
MIXUP_ALPHA = 0.4  # alpha cho Mixup (0 = tắt)
CUTMIX_ALPHA = 1.0  # alpha cho CutMix (0 = tắt)
MIXUP_PROB = 0.3  # xác suất áp dụng Mixup trong 1 batch
CUTMIX_PROB = 0.3  # xác suất áp dụng CutMix trong 1 batch
# Tổng MIXUP_PROB + CUTMIX_PROB = 0.6 → 40% batch train bình thường (không aug)
# Nếu muốn mạnh hơn: tăng lên 0.4 + 0.4

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def format_duration(total_seconds: float) -> str:
    total_seconds = max(0, int(total_seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def build_class_weights(class_counts: list, device: torch.device) -> torch.Tensor:
    """
    Inverse-frequency weighting để xử lý class imbalance.
    battery (975) vs metal (3143) chênh ~3x → cần rebalance.
    Normalize về mean=1 để không ảnh hưởng magnitude của loss.
    """
    counts = torch.tensor(class_counts, dtype=torch.float)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    print("Class weights:")
    for i, w in enumerate(weights):
        print(f"  [{i}] count={class_counts[i]:>5d}  weight={w:.4f}")
    return weights.to(device)


def init_log(path: str, resume: bool):
    if not resume:
        with open(path, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc"])


def log_epoch(path: str, epoch, train_loss, train_acc, val_loss, val_acc):
    with open(path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                epoch,
                f"{train_loss:.6f}",
                f"{train_acc:.6f}",
                f"{val_loss:.6f}",
                f"{val_acc:.6f}",
            ]
        )


def save_checkpoint(
    epoch, model, optimizer, scheduler, scaler, best_val_acc, no_improve
):
    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "best_val_acc": best_val_acc,
            "no_improve": no_improve,
        },
        CHECKPOINT_PATH,
    )


def load_checkpoint(model, optimizer, scheduler, scaler):
    if not os.path.exists(CHECKPOINT_PATH):
        return 0, 0.0, 0
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print(
        f"▶️  Resumed from epoch {ckpt['epoch']} (best_val_acc={ckpt['best_val_acc']:.4f})"
    )
    return ckpt["epoch"], ckpt["best_val_acc"], ckpt["no_improve"]


# ── Mixup / CutMix ────────────────────────────────────────────────────────────
def mixup_data(imgs: torch.Tensor, labels: torch.Tensor, alpha: float):
    """
    Trả về ảnh đã mix, label_a, label_b, lambda.
    loss = lam * loss(pred, a) + (1-lam) * loss(pred, b)
    """
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size = imgs.size(0)
    idx = torch.randperm(batch_size, device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    return mixed, labels, labels[idx], lam


def cutmix_data(imgs: torch.Tensor, labels: torch.Tensor, alpha: float):
    """
    Cắt một vùng chữ nhật ngẫu nhiên từ ảnh B dán vào ảnh A.
    lam thực tế được tính lại theo diện tích vùng cắt.
    """
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    batch_size, _, H, W = imgs.shape
    idx = torch.randperm(batch_size, device=imgs.device)

    # Tính kích thước vùng cắt
    cut_ratio = (1 - lam) ** 0.5
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)

    # Tâm ngẫu nhiên
    cx = random.randint(0, W)
    cy = random.randint(0, H)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y2 = min(cy + cut_h // 2, H)

    mixed = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]

    # Tính lại lam thực tế theo diện tích vùng bị thay
    lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
    return mixed, labels, labels[idx], lam


def mixed_loss(
    criterion: nn.CrossEntropyLoss,
    outputs: torch.Tensor,
    labels_a: torch.Tensor,
    labels_b: torch.Tensor,
    lam: float,
) -> torch.Tensor:
    """
    Loss cho Mixup/CutMix: lam * loss(a) + (1-lam) * loss(b).
    CrossEntropyLoss với weight và label_smoothing vẫn hoạt động đúng
    vì ta truyền hard label riêng lẻ, không phải soft label.
    """
    return lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)


# ── AdamW param groups ────────────────────────────────────────────────────────
def make_param_groups(named_params, lr: float, weight_decay: float) -> list[dict]:
    decay, no_decay = [], []
    for name, p in named_params:
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    groups = []
    if decay:
        groups.append({"params": decay, "lr": lr, "weight_decay": weight_decay})
    if no_decay:
        groups.append({"params": no_decay, "lr": lr, "weight_decay": 0.0})
    return groups


def build_optimizer(
    model: nn.Module,
    lr0: float,
    weight_decay: float,
    variant: str = "resnet101",
) -> optim.AdamW:
    """
    Differential LR theo variant. Tất cả ResNet đều dùng cùng tên layer
    (conv1, bn1, layer1..4, fc) nên chỉ cần thay multiplier.
    """
    assert variant in RESNET_LR_CONFIG, (
        f"Variant '{variant}' chưa có config."
        f"Các variant hỗ trợ: {list(RESNET_LR_CONFIG.keys())}"
    )
    cfg = RESNET_LR_CONFIG[variant]

    stem_named = list(model.conv1.named_parameters()) + list(
        model.bn1.named_parameters()
    )

    param_groups = (
        make_param_groups(stem_named, lr0 * cfg["stem"], weight_decay)
        + make_param_groups(
            model.layer1.named_parameters(), lr0 * cfg["layer1"], weight_decay
        )
        + make_param_groups(
            model.layer2.named_parameters(), lr0 * cfg["layer2"], weight_decay
        )
        + make_param_groups(
            model.layer3.named_parameters(), lr0 * cfg["layer3"], weight_decay
        )
        + make_param_groups(
            model.layer4.named_parameters(), lr0 * cfg["layer4"], weight_decay
        )
        + make_param_groups(model.fc.named_parameters(), lr0 * cfg["fc"], weight_decay)
    )

    # Sanity check: không để sót param nào
    all_grouped = [p for g in param_groups for p in g["params"]]
    all_model = [p for p in model.parameters() if p.requires_grad]
    missed = [p for p in all_model if not any(p is q for q in all_grouped)]
    if missed:
        raise RuntimeError(f"⚠️  {len(missed)} params bị bỏ sót khỏi optimizer!")

    print(
        f"✅ Optimizer [{variant}]: {len(param_groups)} param groups | 0 params bị bỏ sót"
    )
    print(
        f"   LR multipliers → stem={cfg['stem']} | layer1={cfg['layer1']} | "
        f"layer2={cfg['layer2']} | layer3={cfg['layer3']} | "
        f"layer4={cfg['layer4']} | fc={cfg['fc']}"
    )

    return optim.AdamW(param_groups, betas=(0.9, 0.999), eps=1e-8)


# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(IMGSZ, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        transforms.RandomErasing(p=0.25),
    ]
)

eval_transform = transforms.Compose(
    [
        transforms.Resize(IMGSZ + 32),
        transforms.CenterCrop(IMGSZ),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


# ── Model ─────────────────────────────────────────────────────────────────────
def build_model(
    num_classes: int, dropout: float = DROPOUT, variant: str = "resnet101"
) -> nn.Module:
    weights_map = {
        "resnet18": models.ResNet18_Weights.DEFAULT,
        "resnet34": models.ResNet34_Weights.DEFAULT,
        "resnet50": models.ResNet50_Weights.DEFAULT,
        "resnet101": models.ResNet101_Weights.DEFAULT,
        "resnet152": models.ResNet152_Weights.DEFAULT,
    }
    builder_map = {
        "resnet18": models.resnet18,
        "resnet34": models.resnet34,
        "resnet50": models.resnet50,
        "resnet101": models.resnet101,
        "resnet152": models.resnet152,
    }

    assert variant in builder_map, f"Variant không hợp lệ: {variant}"
    model = builder_map[variant](weights=weights_map[variant])

    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes),
    )
    return model


# ── Train / Eval loops ────────────────────────────────────────────────────────
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    accumulation_steps,
    epoch,
    total_epochs,
):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    optimizer.zero_grad()

    pbar = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=f"Epoch {epoch:03d}/{total_epochs} [Train]",
        dynamic_ncols=True,
    )

    for step, (imgs, labels) in pbar:
        imgs, labels = imgs.to(device), labels.to(device)

        # ── Chọn augmentation cho batch này ───────────────────────────────────
        r = random.random()
        if r < CUTMIX_PROB:
            imgs, labels_a, labels_b, lam = cutmix_data(imgs, labels, CUTMIX_ALPHA)
            use_mixed = True
        elif r < CUTMIX_PROB + MIXUP_PROB:
            imgs, labels_a, labels_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
            use_mixed = True
        else:
            use_mixed = False

        with autocast(device_type="cuda"):
            outputs = model(imgs)
            if use_mixed:
                loss = (
                    mixed_loss(criterion, outputs, labels_a, labels_b, lam)
                    / accumulation_steps
                )
            else:
                loss = criterion(outputs, labels) / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * accumulation_steps * imgs.size(0)
        # Accuracy: với mixed batch dùng label gốc để tính (approximate)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)

        pbar.set_postfix(
            {
                "loss": f"{running_loss / total:.4f}",
                "acc": f"{correct / total:.4f}",
            }
        )

    # Flush gradient nếu còn dư
    if (step + 1) % accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch, total_epochs, split="Val"):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(
        loader,
        total=len(loader),
        desc=f"Epoch {epoch:03d}/{total_epochs} [{split}]",
        dynamic_ncols=True,
    )

    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast(device_type="cuda"):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)

        pbar.set_postfix(
            {
                "loss": f"{running_loss / total:.4f}",
                "acc": f"{correct / total:.4f}",
            }
        )

    return running_loss / total, correct / total


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    os.makedirs(save_path, exist_ok=True)
    torch.backends.cudnn.benchmark = True
    os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

    # Datasets & Loaders
    train_dataset = datasets.ImageFolder(
        os.path.join(dataset_root, "train"), transform=train_transform
    )
    val_dataset = datasets.ImageFolder(
        os.path.join(dataset_root, "val"), transform=eval_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True,
    )

    class_names = train_dataset.classes
    num_classes = len(class_names)
    print(f"Classes ({num_classes}): {class_names}")
    print(
        f"Device: {DEVICE} | Variant: {RESNET_VARIANT} | Effective batch: {BATCH * ACCUMULATION_STEPS}"
    )
    print(
        f"Mixup(α={MIXUP_ALPHA}, p={MIXUP_PROB}) | "
        f"CutMix(α={CUTMIX_ALPHA}, p={CUTMIX_PROB}) | "
        f"Normal batch: {1 - MIXUP_PROB - CUTMIX_PROB:.0%}"
    )

    # Kiểm tra thứ tự class
    expected_classes = [
        "battery",
        "biological",
        "cardboard",
        "clothes",
        "e-waste",
        "glass",
        "metal",
        "paper",
        "plastic",
        "shoes",
    ]
    if class_names != expected_classes:
        print("⚠️  Thứ tự class KHÁC expected — CLASS_COUNTS có thể sai!")
        print(f"   Detected : {class_names}")
        print(f"   Expected : {expected_classes}")
    else:
        print("✅ Thứ tự class khớp CLASS_COUNTS")

    # Model
    model = build_model(num_classes, dropout=DROPOUT, variant=RESNET_VARIANT).to(DEVICE)

    # Loss với class weights
    class_weights = build_class_weights(CLASS_COUNTS, DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

    # Optimizer
    optimizer = build_optimizer(
        model, lr0=LR0, weight_decay=WEIGHT_DECAY, variant=RESNET_VARIANT
    )

    # Scheduler: Linear warmup → Cosine annealing
    warmup = LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS
    )
    cosine = CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS - WARMUP_EPOCHS,
        eta_min=0,
    )
    scheduler = SequentialLR(
        optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS]
    )
    scaler = GradScaler()

    # Resume nếu có checkpoint
    start_epoch, best_val_acc, no_improve = load_checkpoint(
        model, optimizer, scheduler, scaler
    )
    init_log(LOG_PATH, resume=start_epoch > 0)

    epoch_pbar = tqdm(
        range(start_epoch + 1, EPOCHS + 1), desc="📦 Training", dynamic_ncols=True
    )
    start_time = time.perf_counter()

    for epoch in epoch_pbar:
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler,
            DEVICE,
            ACCUMULATION_STEPS,
            epoch,
            EPOCHS,
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, DEVICE, epoch, EPOCHS, split="Val"
        )
        scheduler.step()

        log_epoch(LOG_PATH, epoch, train_loss, train_acc, val_loss, val_acc)

        elapsed = format_duration(time.perf_counter() - start_time)
        epoch_pbar.set_postfix(
            {
                "train_acc": f"{train_acc:.4f}",
                "val_acc": f"{val_acc:.4f}",
                "best": f"{best_val_acc:.4f}",
                "patience": f"{no_improve}/{PATIENCE}",
                "elapsed": elapsed,
            }
        )

        save_checkpoint(
            epoch, model, optimizer, scheduler, scaler, best_val_acc, no_improve
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                tqdm.write(f"  ⏹️ Early stopping at epoch {epoch}")
                break